In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psutil
import platform
import os

from tensorflow.keras.datasets import cifar100
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Dense,
    Dropout,
    BatchNormalization,
    Activation,
    GlobalAveragePooling2D
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)
from sklearn.metrics import confusion_matrix, classification_report

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
(x_train, y_train), (x_test, y_test) = cifar100.load_data()

print("X_train:", x_train.shape)
print("y_train:", y_train.shape)
print("X_test :", x_test.shape)
print("y_test :", y_test.shape)
print("\nNúmero de classes:", len(np.unique(y_train)))

X_train: (50000, 32, 32, 3)
y_train: (50000, 1)
X_test : (10000, 32, 32, 3)
y_test : (10000, 1)

Número de classes: 100


In [ ]:
# Normalização dos pixels (escala de 0 a 1)
X_train = x_train.astype("float32") / 255.0
X_test = x_test.astype("float32") / 255.0

# One-Hot Encoding para as 100 classes
y_train_cat = to_categorical(y_train, 100)
y_test_cat = to_categorical(y_test, 100)

# Separação de validação (reservando 5000 imagens)
X_val = X_train[:5000]
y_val = y_train_cat[:5000]

X_train_f = X_train[5000:]
y_train_f = y_train_cat[5000:]

print("Treino:", X_train_f.shape, y_train_f.shape)
print("Validação:", X_val.shape, y_val.shape)
print("Teste:", X_test.shape, y_test_cat.shape)

Treino: (45000, 32, 32, 3) (45000, 100)
Validação: (5000, 32, 32, 3) (5000, 100)
Teste: (10000, 32, 32, 3) (10000, 100)


In [8]:
datagen = ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

datagen.fit(X_train_f)
print("Data Augmentation configurado com sucesso!")

Data Augmentation configurado com sucesso!


In [ ]:
model = Sequential()

# ===== BLOCO 1 =====
model.add(Conv2D(64, (3,3), padding='same', input_shape=(32,32,3)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv2D(64, (3,3), padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))

# ===== BLOCO 2 =====
model.add(Conv2D(128, (3,3), padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv2D(128, (3,3), padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))

# ===== BLOCO 3 =====
model.add(Conv2D(256, (3,3), padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv2D(256, (3,3), padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.35))

# ===== BLOCO 4 =====
model.add(Conv2D(512, (3,3), padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv2D(512, (3,3), padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.35))

# ===== CLASSIFICADOR =====
model.add(GlobalAveragePooling2D())
model.add(Dense(512))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(100, activation='softmax'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 8, 8, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 8, 8, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 8, 8, 256)      │             

 Total params: 5,009,060 (19.11 MB)

 Trainable params: 5,004,196 (19.09 MB)

 Non-trainable params: 4,864 (19.00 KB)

In [9]:
# A V3 utiliza learning_rate = 0.001
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

checkpoint_v3 = ModelCheckpoint(
    'modelo_v3.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

print("Modelo compilado e callbacks configurados (Específico V3).")

Modelo compilado e callbacks configurados (Específico V3).


In [10]:
history_v3 = model.fit(
    datagen.flow(
        X_train_f,
        y_train_f,
        batch_size=32
    ),
    # A linha steps_per_epoch foi APAGADA!
    validation_data=(X_val, y_val),
    epochs=150,
    callbacks=[
        early_stop,
        reduce_lr,
        checkpoint_v3
    ],
    verbose=1
)


Epoch 1/150
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.0422 - loss: 4.3301
Epoch 1: val_accuracy improved from None to 0.11060, saving model to modelo_v3.keras

Epoch 1: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 51s 26ms/step - accuracy: 0.0612 - loss: 4.1303 - val_accuracy: 0.1106 - val_loss: 3.7230 - learning_rate: 0.0010
Epoch 2/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1133 - loss: 3.7484
Epoch 2: val_accuracy improved from 0.11060 to 0.16480, saving model to modelo_v3.keras

Epoch 2: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 26s 19ms/step - accuracy: 0.1258 - loss: 3.6747 - val_accuracy: 0.1648 - val_loss: 3.4412 - learning_rate: 0.0010
Epoch 3/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1684 - loss: 3.4327
Epoch 3: val_accuracy improved from 0.16480 to 0.21120, saving model to modelo_v3.keras

Epoch 3: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━